# BirdCLEF 2026 — Inference Notebook

**Setup:**
1. Upload **`best_model.keras`** (from `models/` or `runs/<run_id>/`) to your `birdclef-model` dataset
2. Add the `birdclef-model` dataset via **Add Data → Your Datasets**
3. **Save Version** (Run All) → writes a valid `submission.csv` even though `test_soundscapes/` is empty in the public run
4. **Submit** the committed version → Kaggle privately re-runs with the real hidden soundscapes and scores that output

**Version-safe:** local `keras_env` is pinned to Kaggle's exact TF 2.19.0 / Keras 3.10.0, so any model the agent saves loads here with no shim and no architecture editing — regardless of which head that run produced. Cell 3's smoke test fails loudly in the public commit if anything is wrong, instead of silently scoring ~0.5.

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf

print('TF version:', tf.__version__)
print('Devices:', tf.config.list_physical_devices())

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
import glob

COMPETITION_DIR = '/kaggle/input/competitions/birdclef-2026/'

# Find the model anywhere under /kaggle/input, instead of hardcoding a path that
# breaks if the dataset slug / filename / nesting differs. Prefers best_model.keras,
# then any .keras file.
def find_model():
    preferred = glob.glob('/kaggle/input/**/best_model.keras', recursive=True)
    anykeras  = glob.glob('/kaggle/input/**/*.keras', recursive=True)
    hits = preferred or anykeras
    if not hits:
        listing = glob.glob('/kaggle/input/**/*', recursive=True)
        raise FileNotFoundError(
            'No .keras model found under /kaggle/input. '
            'Did you add the birdclef-model dataset? Contents:\n  '
            + '\n  '.join(listing[:60])
        )
    return sorted(hits)[0]

MODEL_PATH = find_model()
print('Using model:', MODEL_PATH)

TEST_DIR        = COMPETITION_DIR + 'test_soundscapes/'
TAXONOMY_PATH   = COMPETITION_DIR + 'taxonomy.csv'
SAMPLE_SUB_PATH = COMPETITION_DIR + 'sample_submission.csv'

# ── Audio config — MUST match agent.py exactly ────────────────────────────────
SAMPLE_RATE = 32000
DURATION    = 5
N_MELS      = 64    # must equal agent.py N_MELS (spectrogram frequency bins)
F_MAX       = 16000
BATCH_SIZE  = 32

In [ ]:
# ── Load the full trained model (architecture + weights together) ─────────────
# keras_env was pinned to Kaggle's exact TF 2.19.0 / Keras 3.10.0, so the
# saved .keras config contains no keys Kaggle's Keras can't parse. No shim, no
# hardcoded head — works for whatever architecture the agent's best run used.
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print('Model loaded from', MODEL_PATH)

# Smoke test — fails loudly in the public commit if anything is wrong,
# instead of silently producing a ~0.5 submission. Input freq dim = N_MELS.
_dummy = np.zeros((2, N_MELS, 313, 1), dtype=np.float32)
_p = model.predict(_dummy, verbose=0)
assert _p.shape == (2, 234), f'Unexpected output shape {_p.shape} (expected (2, 234))'
print(f'Smoke test OK — output {_p.shape}, range [{_p.min():.4f}, {_p.max():.4f}]')

In [ ]:
# ── Load taxonomy — defines species column order ──────────────────────────────
# Cast to str so column names match the string headers in sample_submission.csv
taxonomy = pd.read_csv(TAXONOMY_PATH)
species  = taxonomy['primary_label'].astype(str).tolist()  # 234 species, must be strings
print(f'Species: {len(species)}')

# Verify against sample submission
sample_sub     = pd.read_csv(SAMPLE_SUB_PATH)
sub_species    = sample_sub.columns[1:].tolist()
assert species == sub_species, f'Species mismatch: taxonomy has {species[:3]}, submission has {sub_species[:3]}'
print('Species order verified against sample_submission.')

In [ ]:
# ── Audio helpers — identical to agent.py ────────────────────────────────────
def audio_to_melspectrogram(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, fmax=F_MAX
    )
    return librosa.power_to_db(mel, ref=np.max)


def extract_windows(filepath):
    """Load a soundscape and return list of (end_second, spectrogram) tuples."""
    try:
        audio, _ = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f'  Failed to load {filepath}: {e}')
        return []

    windows = []
    step    = SAMPLE_RATE * DURATION
    total   = len(audio)

    for start in range(0, total, step):
        chunk = audio[start:start + step]
        if len(chunk) < step:
            chunk = np.pad(chunk, (0, step - len(chunk)))
        mel        = audio_to_melspectrogram(chunk)
        end_second = start // SAMPLE_RATE + DURATION
        windows.append((end_second, mel))

    return windows

In [ ]:
# ── Run inference on all test soundscapes ────────────────────────────────────
# Walk recursively in case files are in subdirectories
test_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(TEST_DIR)
    for f in files if f.endswith('.ogg')
])
print(f'Test files: {len(test_files)}')

rows = []

for i, fpath in enumerate(test_files):
    stem    = os.path.splitext(os.path.basename(fpath))[0]
    windows = extract_windows(fpath)

    if not windows:
        continue

    end_seconds = [w[0] for w in windows]
    specs       = np.array([w[1] for w in windows])[..., np.newaxis]

    for batch_start in range(0, len(specs), BATCH_SIZE):
        batch      = specs[batch_start:batch_start + BATCH_SIZE]
        preds      = model.predict(batch, verbose=0)
        batch_ends = end_seconds[batch_start:batch_start + BATCH_SIZE]

        for end_sec, pred in zip(batch_ends, preds):
            row = {'row_id': f'{stem}_{end_sec}'}
            row.update(dict(zip(species, pred.tolist())))
            rows.append(row)

    if (i + 1) % 10 == 0:
        print(f'  Processed {i + 1}/{len(test_files)} files')

print(f'Total rows: {len(rows)}')

In [ ]:
# ── Build submission ──────────────────────────────────────────────────────────
if rows:
    submission = pd.DataFrame(rows)[['row_id'] + species]
    vals = submission[species].values
    print(f'Prediction stats — min={vals.min():.4f} max={vals.max():.4f} '
          f'mean={vals.mean():.4f}')
    if vals.max() == vals.min():
        print('WARNING: predictions are constant — model output is degenerate')
else:
    # No test files: EXPECTED during the public Save Version commit
    # (test_soundscapes/ holds only readme.txt then). During Kaggle's private
    # re-run the real soundscapes are present, so this branch is NOT taken and
    # the model above does the real predicting.
    print('No test files found — public dry-run. Writing a zero-filled '
          'submission so the commit succeeds; the private re-run predicts for real.')
    submission = sample_sub.copy()
    submission[species] = 0.0

submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved — {len(submission)} rows x {len(submission.columns)} columns')
submission.head(3)